# AXI test 16: the serving side a third time, with the length of a name capped
**Why this test exists.** In the second run (test 15) every reply of both arms had to match the repository's grammar. One reply of the exact-state arm (job 19, step 12) then ran on inside a *name* until the 96-token limit, because that grammar lets a name be any length. By the page's own rule that made reading line 2 a NO, and the figures it held back (exact steps per GPU hour) were not claimed. This run **caps a name at 16 characters** and runs the exact-state cells again (`S` with 8, 32 and 64 jobs) and the kept chat at 8 jobs. The kept chat at 32 and 64 jobs is **not** run again: its cells are read from the second run's file. The predictions and the reading rules were written before this run in `docs/PROVIDER_SIDE_BOUNDED_TEST.md`.

Model: **Qwen2.5-7B-Instruct, 4-bit AWQ**, engine **vLLM 0.31.0**, cache fixed at **80,000 tokens**, GPU **T4**.

**Before you start:** Runtime menu -> *Change runtime type* -> **T4 GPU** -> Save. Run each cell **one at a time, top to bottom**, wait for the green tick before the next one. **Keep this browser tab open** while a cell runs.

**How long it takes (a guess from the second run, a free T4):** installing the engine about 5 minutes, loading the model 5 to 10 minutes, the safety check 3 minutes, the four cells about 10 minutes. In all about 25 to 30 minutes.

**If Colab stops** (the free GPU time runs out, the tab closes): reconnect, run Cells 1 to 5 again (do not forget Cell 4b), and then run the cell you were in. Finished cells are skipped; a cell that was running continues from its last finished round.

Do not paste any keys anywhere.

### Cell 1: check the GPU (it must say Tesla T4, or another GPU with at least 15 GB)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

### Cell 2: get the code from GitHub. If it stops with 'the code is not on GitHub yet', finish the update 60 commands in Codespaces (git push) and run this cell again

In [ ]:
!rm -rf /content/axi && git clone -q --depth 1 https://github.com/gaurshalendra2412/axi-runtime /content/axi
import os
assert os.path.exists("/content/axi/python/axi/experiments/provider_side_bounded.py"), "the code is not on GitHub yet: finish the update 60 commands (git push) first"
assert os.path.exists("/content/axi/docs/results/axi_provider_constrained.json"), "the second run's file is not on GitHub"
print("code ok")

### Cell 3: install the engine, version 0.31.0 (about 5 minutes). If Colab says 'Restart session', press it, then run Cells 2, 3, 4 and 5 again (Cell 3 will then be quick)

In [ ]:
!pip -q install vllm==0.31.0

### Cell 4: connect Google Drive (a pop-up asks permission: tap Connect / Allow). If it prints 'Drive NOT connected', the run still works but nothing survives a disconnect

In [ ]:
import sys, os, json, time, glob, re
sys.path.insert(0, "/content/axi/python")
from colab.colab_persist import open_store
store = open_store("/content/out", "axi_provider_bounded")

### Cell 4b: fix the output stream (about 1 second). vLLM 0.31.0 asks the notebook for a file number of its output and a Colab cell has none, so Cell 5 stops with `io.UnsupportedOperation: fileno`. This cell gives the notebook's output the usual numbers 1 and 2. Run it once after every restart, before Cell 5

In [ ]:
import sys
for name, fd in (("stdout", 1), ("stderr", 2)):
    s = getattr(sys, name)
    try: s.fileno()
    except Exception: s.fileno = (lambda fd=fd: fd)
print("output numbers:", sys.stdout.fileno(), sys.stderr.fileno())

### Cell 5: load the model and the engine with the capped constraint (downloads about 5.5 GB once; about 5 to 10 minutes). It prints the engine's own numbers. If it fails with a red error, stop and send me the red text (if the red text says `fileno`, run Cell 4b first; if it says `StructuredOutputsParams`, send it to me and do nothing else)

In [ ]:
from axi.experiments import provider_side_bounded as B
eng = B.BoundedVLLMEngine()
info = eng.info(); print(json.dumps({k: v for k, v in info.items() if k != "constraint"}, indent=1))
print("constraint:", info["constraint"]["kind"], "name at most", info["constraint"]["name_cap"], "characters", info["constraint"]["regex_sha256"][:16], "(the same for both arms)")
eng.warm_up()
print("cache size the engine will use:", info["capacity_tokens"], "tokens (", info["capacity_source"], ")")

def go(arm, K):
    """Runs one cell (arm, K) unless its file is already there; carries on from its checkpoint if the last session ended inside it; saves it and copies it to Drive."""
    name = f"axi_provider_b_{arm}_K{K}.json"; ck = f"axi_provider_b_{arm}_K{K}.ckpt.json"
    print(f"=== {arm} K={K}: {store.restore(name)}")
    if os.path.exists(store.path(name)): print("   already finished, skipping"); return
    print(f"   checkpoint: {store.restore(ck)}")
    if os.path.exists(store.path(ck)): print(f"   carrying on from round {len(json.load(open(store.path(ck)))['rounds'])}")
    t0 = time.time()
    def progress(i, a, k): store.persist(ck); print(f"   {a} K={k}  round {i}   {time.time() - t0:6.0f} s", end="\r")
    c = B.run_cell(eng, arm, K, progress=progress, checkpoint=store.path(ck))
    B.save_cell(c, store.path(name)); store.persist(name)
    s = B.summarize_cell2(c); f = lambda x, d=1: "-" if x is None else f"{x:.{d}f}"
    print(f"\n   done: {s['steps_done']} steps, {s['wall_seconds']:.0f} s, {f(s['seconds_per_step'], 2)} s/step, computed prompt tokens {f(s['computed_prefill'], 0)}, cached share {f(s['cached_share'] and s['cached_share'] * 100, 0)} %, exact steps {s['exact_share'] * 100:.1f} %, refused replies {s['unparsed']}, exact steps per hour {f(s['exact_per_hour'], 0)}" + (f", resumed at rounds {s['resumed_at']}" if s["resumed_at"] else ""))

### Cell 6: SAFETY CHECK (about 3 minutes). 3 jobs of 3 steps in each of S and KC, then ONE probe: the exact-state prompt of step 12 of job 19, where the second run's engine answered `ADD[(3,0)->(3,0):4tworowsdown4colsright...` until the 96-token limit. It prints the sizes, the cached tokens and the probe's reply. If it says STOP, do not run the next cells: send me the printout

In [ ]:
smoke = {arm: B.run_cell(eng, arm, 3, steps=3) for arm in ("S", "KC")}
for arm, c in smoke.items():
    for x in c["sessions"][0]["log"]: print(f"{arm:2s} step {x['t']}: prompt {x['n_prompt']:5d}  cached {x['n_cached']}  new {x['n_new']:3d}  {x['text'][:50]!r}")
steps = [x for c in smoke.values() for s in c["sessions"] for x in s["log"]]
ok_cached = all(x["n_cached"] is not None for x in steps)
kc = smoke["KC"]["sessions"][0]["log"]
ok_work = ok_cached and kc[1]["n_cached"] > 0
ok_parse = all(x["parsed"] for x in steps)
probe = B.loop_probe(eng)
ok_in_language = re.fullmatch(B.REGEX, probe["reply"]) is not None
print(f"probe (job {probe['episode']}, step {probe['step']}, {probe['kind']}): the second run's engine ran on to 96 tokens; now the engine answers {probe['reply']!r} ({probe['n_new']} tokens); the strict parser {'reads it' if probe['parsed'] else 'REFUSES it'}; inside the capped language: {ok_in_language}")
print()
print("STOP: this vLLM version does not report cached tokens. Send me the printout and the version above." if not ok_cached else
      "STOP: the cache did nothing for the kept chat's second step. Send me the printout." if not ok_work else
      "STOP: a reply of the safety cell was refused by the strict parser, so the constraint is not working. Send me the printout." if not ok_parse else
      "STOP: the probe's reply is refused by the strict parser. Send me the printout." if not probe["parsed"] else
      "STOP: the probe's reply used all 96 new tokens, so the name is not capped. Send me the printout." if probe["n_new"] >= 96 else
      "STOP: the probe's reply is not inside the capped language. Send me the printout." if not ok_in_language else
      "OK: cached counts are reported, the cache works, every reply is well formed and the probe's reply ends. You can go on.")

### Cell 7: the four cells, in this order: S with 8 jobs, KC with 8 jobs, S with 32 jobs, S with 64 jobs (about 10 minutes in all). A counter shows the round

In [ ]:
for arm, K in (("S", 8), ("KC", 8), ("S", 32), ("S", 64)): go(arm, K)

### Cell 8: put the finished cells together, read the second run's file from the repository, print the table and the 13 reading-rule lines, and download your file. You can run it at any time, with whatever cells are finished

In [ ]:
from google.colab import files
SECOND = "/content/axi/docs/results/axi_provider_constrained.json"
paths = sorted(glob.glob(store.path("axi_provider_b_*_K*.json")))
paths = [p for p in paths if not p.endswith(".ckpt.json")]
print(len(paths), "cells:", [os.path.basename(p) for p in paths])
m = B.merge_files(paths, store.path("axi_provider_bounded.json"), second_run=SECOND); store.persist("axi_provider_bounded.json")
old = B.load_second_run(SECOND)["cells"]
S = B.summarize_all(m["cells"]); B.print_cells(S); B.print_verdict(B.verdict_bounded(m["cells"], old))
files.download(store.path("axi_provider_bounded.json"))

### Done
File downloaded: `axi_provider_bounded.json`. Upload it to the chat, with the cell files from your Drive folder `axi_provider_bounded` (the `axi_provider_b_*.json` files, not the `.ckpt.json` ones), a screenshot of the printed table and of the 13 reading-rule lines, and the engine numbers printed by Cell 5 (vLLM version, GPU, cache size). Do not paste any keys anywhere.